# PPA EDA - Phase 2 Week 3

**Goal**: distributions, pack segmentation, elasticity signals, seasonality.

Follows the engineered features built in `src/features.py`; no data is modified here.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    import seaborn as sns
    sns.set_theme(style='whitegrid')
except ImportError:
    sns = None

from src.features import add_base_features, add_panel_features
from src.split import expanding_window_cv, final_holdout_split, describe_folds
from src.config import DATA_PATH, CANDIDATE_FEATURES, CATEGORICAL_COLS

In [ ]:
df = pd.read_csv(DATA_PATH)
print('shape:', df.shape)
df.head()

In [ ]:
df_b = add_base_features(df)
df_fe = add_panel_features(df_b, df_b)
print('after feature engineering:', df_fe.shape)
print('new columns:', [c for c in df_fe.columns if c not in df.columns])

## Target distribution (right-skewed -> log transform mandatory)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df_fe['nielsen_total_volume'].hist(bins=80, ax=axes[0])
axes[0].set_title('Volume (raw, highly right-skewed)')
df_fe['log_nielsen_total_volume'].hist(bins=80, ax=axes[1])
axes[1].set_title('log1p(volume)')
plt.show()

## Time-series CV fold summary

In [ ]:
dev_idx, test_idx = final_holdout_split(df_fe)
df_dev = df_fe.iloc[dev_idx].reset_index(drop=True)
print('dev rows:', len(df_dev), '  sealed test rows:', len(test_idx))
describe_folds(df_dev)

## Price-volume scatter by pack tier (elasticity signal)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for tier, sub in df_fe.groupby('pack_tier'):
    ax.scatter(sub['log_price_per_litre'], sub['log_nielsen_total_volume'], s=3, alpha=0.15, label=tier)
ax.set_xlabel('log(price_per_litre)'); ax.set_ylabel('log1p(volume)')
ax.set_title('Negative slope is the elasticity signal we want to fit')
ax.legend(markerscale=3)
plt.show()

## Promo intensity vs volume

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
df_fe.boxplot(column='log_nielsen_total_volume', by='promotion_indicator', ax=ax)
ax.set_title('log_nielsen_total_volume by promotion_indicator'); ax.set_xlabel('promo'); ax.set_ylabel('log_nielsen_total_volume')
plt.suptitle('')
plt.show()

## Correlation map (sanity check on collinearity claim)

In [ ]:
num_cols = ['price_per_litre', 'log_price_per_litre', 'pack_size_total',
            'pack_size_internal', 'units_per_package_internal',
            'promotion_indicator', 'promo_depth',
            'price_premium_vs_brand', 'price_premium_vs_pack_tier',
            'log_nielsen_total_volume_lag1', 'log_nielsen_total_volume_lag4', 'log_nielsen_total_volume']
corr = df_fe[num_cols].corr()
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(num_cols))); ax.set_xticklabels(num_cols, rotation=70)
ax.set_yticks(range(len(num_cols))); ax.set_yticklabels(num_cols)
plt.colorbar(im, ax=ax); plt.tight_layout(); plt.show()